In [1]:
import numpy as np


class SimpleLinearRegressor:
  """Level-1 Meta-Learner or Level-0 Base Linear Model with L2 penalty (Ridge)."""

  def __init__(self, alpha=1.0):
    self.alpha = alpha
    self.weights = None
    self.bias = 0.0

  def fit(self, X, y):
    n_samples, n_features = X.shape
    X_b = np.hstack([np.ones((n_samples, 1)), X])
    I = np.eye(n_features + 1)
    I[0, 0] = 0.0  # Do not regularize intercept

    # Closed-form Ridge: w = (X^T X + alpha * I)^(-1) X^T y
    theta = np.linalg.inv(X_b.T @ X_b + self.alpha * I) @ (X_b.T @ y)
    self.bias = theta[0]
    self.weights = theta[1:]

  def predict(self, X):
    return X @ self.weights + self.bias


class SimpleDecisionStumpRegressor:
  """Level-0 Weak Tree Base Estimator."""

  def __init__(self):
    self.feature_idx = None
    self.threshold = None
    self.left_val = 0.0
    self.right_val = 0.0

  def fit(self, X, y):
    best_error = float("inf")
    n_samples, n_features = X.shape

    for feat in range(n_features):
      thresholds = np.unique(X[:, feat])
      for th in thresholds:
        left_mask = X[:, feat] <= th
        right_mask = ~left_mask

        if np.sum(left_mask) == 0 or np.sum(right_mask) == 0:
          continue

        left_m = np.mean(y[left_mask])
        right_m = np.mean(y[right_mask])

        err = np.sum((y[left_mask] - left_m) ** 2) + np.sum(
            (y[right_mask] - right_m) ** 2
        )

        if err < best_error:
          best_error = err
          self.feature_idx = feat
          self.threshold = th
          self.left_val = left_m
          self.right_val = right_m

  def predict(self, X):
    mask = X[:, self.feature_idx] <= self.threshold
    preds = np.empty(X.shape[0])
    preds[mask] = self.left_val
    preds[~mask] = self.right_val
    return preds


class StackingRegressorScratch:
  """Two-Level Stacking Regressor using K-Fold Out-of-Fold (OOF) Prediction Matrix."""

  def __init__(self, base_models, meta_model, n_folds=5):
    self.base_models = base_models
    self.meta_model = meta_model
    self.n_folds = n_folds
    self.fitted_base_models = []

  def fit(self, X, y):
    n_samples = X.shape[0]
    n_base = len(self.base_models)

    # Matrix Z holds out-of-fold predictions to prevent leakage
    oof_predictions = np.zeros((n_samples, n_base))

    # K-Fold generation
    indices = np.arange(n_samples)
    np.random.shuffle(indices)
    folds = np.array_split(indices, self.n_folds)

    # 1. Generate unbiased OOF meta-features
    for fold_idx, val_idx in enumerate(folds):
      train_idx = np.setdiff1d(indices, val_idx)
      X_train_f, y_train_f = X[train_idx], y[train_idx]
      X_val_f = X[val_idx]

      for model_idx, model_factory in enumerate(self.base_models):
        fold_model = model_factory()
        fold_model.fit(X_train_f, y_train_f)
        oof_predictions[val_idx, model_idx] = fold_model.predict(X_val_f)

    # 2. Train the Level-1 Meta-Learner on the OOF Matrix
    self.meta_model.fit(oof_predictions, y)

    # 3. Fit base models on the entire dataset for inference
    self.fitted_base_models = []
    for model_factory in self.base_models:
      full_model = model_factory()
      full_model.fit(X, y)
      self.fitted_base_models.append(full_model)

  def predict(self, X):
    n_samples = X.shape[0]
    meta_features = np.zeros((n_samples, len(self.fitted_base_models)))

    # Collect predictions from each Level-0 base model
    for idx, model in enumerate(self.fitted_base_models):
      meta_features[:, idx] = model.predict(X)

    # Level-1 Meta-Learner makes final consensus decision
    return self.meta_model.predict(meta_features)


# =====================================================================
# DEMO EXECUTION
# =====================================================================
if __name__ == "__main__":
  np.random.seed(42)

  # Synthetic non-linear data
  X = np.linspace(-3, 3, 120).reshape(-1, 1)
  y = np.sin(X).ravel() + 0.3 * X.ravel() + np.random.normal(0, 0.1, X.shape[0])

  # Base estimators: Diverse inductive biases
  base_learners = [
      lambda: SimpleLinearRegressor(alpha=1.0),
      lambda: SimpleDecisionStumpRegressor(),
  ]

  # Meta learner: Ridge Regressor
  meta_learner = SimpleLinearRegressor(alpha=0.5)

  # Train Stacking Ensemble
  ensemble = StackingRegressorScratch(
      base_models=base_learners, meta_model=meta_learner, n_folds=5
  )
  ensemble.fit(X, y)
  preds = ensemble.predict(X)

  mse = np.mean((y - preds) ** 2)
  print("=" * 65)
  print("  TWO-LEVEL STACKING ENSEMBLE FROM SCRATCH (NUMPY)")
  print("=" * 65)
  print(f"Total Base Learners (Level 0) : {len(base_learners)}")
  print(f"Meta-Learner (Level 1)        : Ridge Meta-Model")
  print(f"OOF Cross-Validation Folds    : {ensemble.n_folds}")
  print(f"Ensemble Mean Squared Error   : {mse:.6f}")
  print("=" * 65)

  TWO-LEVEL STACKING ENSEMBLE FROM SCRATCH (NUMPY)
Total Base Learners (Level 0) : 2
Meta-Learner (Level 1)        : Ridge Meta-Model
OOF Cross-Validation Folds    : 5
Ensemble Mean Squared Error   : 0.094847
